## 데이터 로드 / 셋업

In [1]:
from collections import defaultdict, deque
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 160)

ROOT = Path.cwd()
TRAIN_PATH = ROOT / 'train_cleaned.csv'
TEST_PATH = ROOT / 'test_cleaned.csv'
OUT_DIR = ROOT / 'output'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 저장
TRAIN_OUT = OUT_DIR / 'train_eda.csv'
TEST_OUT = OUT_DIR / 'test_eda.csv'

# 날짜 계산에 사용할 날짜 컬럼
DATE_COLS = ['baseline_create_date', 'due_in_date', 'clear_date']

FEATURE_COLUMNS = [
    'business_days_late',
    'target',
    'due_weekend_flag',
    'business_day_gap',
    'amount_bin',
    'cust_allowed_pay_days_late_rate_past',
    'ratio_paid_invoices_late_past',
    'avg_days_late_paid_late_past',
    'sum_outstanding_amount_past',
    'recent_5_late_rate',
]

dtype = {
    'cust_number': 'string',
    'business_code': 'string',
    'name_customer': 'string',
    'cust_payment_terms': 'string',
    'cust_payment_terms_grp': 'string',
}

# 기존 target은 target_old로
# 새 target은 새 기준으로 다시 생성
train_raw = pd.read_csv(TRAIN_PATH, dtype=dtype).rename(columns={'target': 'target_old'})
test_raw = pd.read_csv(TEST_PATH, dtype=dtype).rename(columns={'target': 'target_old'})

train_featured = train_raw.copy()
test_featured = test_raw.copy()

# datetime
for df in [train_featured, test_featured]:
    for col in DATE_COLS:
        df[col] = pd.to_datetime(df[col], errors='raise')

# 확인
for name, df in [('train', train_featured), ('test', test_featured)]:
    print(f'{name}: rows={len(df):,}, cols={len(df.columns)}, customers={df["cust_number"].nunique():,}')
    print('target_old counts:', df['target_old'].value_counts(dropna=False).sort_index().to_dict())
    print('missing values:', int(df.isna().sum().sum()))
    display(df.head(3)) if 'display' in globals() else print(df.head(3).to_string(index=False))


train: rows=32,000, cols=16, customers=981
target_old counts: {0: 18393, 1: 13607}
missing values: 0
business_code cust_number   name_customer clear_date  buisness_year due_in_date  posting_id baseline_create_date cust_payment_terms  target_old  amount_in_usd cust_payment_terms_grp  baseline_month  baseline_day  baseline_dayofweek  Allowed_Pay_Days
         U001  0200706844         WINC co 2019-02-19         2019.0  2018-12-29         1.0           2018-12-14               NAA8           1       6.359366                   NAA8              12            14                   4                15
         CA02  0140104249  SOB associates 2019-01-23         2019.0  2018-12-24         1.0           2018-12-14               CA10           1      11.663576                   CA10              12            14                   4                10
         U001  0200769623 WAL-MAR systems 2019-01-09         2019.0  2019-01-14         1.0           2018-12-30               NAH4           0      

## business_days_late


In [2]:
# 만기일과 실제 결제일 사이의 지연일을 영업일 기준으로 계산
def add_business_days_late(df: pd.DataFrame) -> None:
    # np.busday_count
    due_days = df['due_in_date'].values.astype('datetime64[D]')
    clear_days = df['clear_date'].values.astype('datetime64[D]')

    # due_in_date부터 clear_date까지 영업일
    df['business_days_late'] = np.busday_count(due_days, clear_days).astype(int)

add_business_days_late(train_featured)
add_business_days_late(test_featured)

print(train_featured['business_days_late'].describe())

count    32000.000000
mean         0.618437
std          7.928335
min        -63.000000
25%         -2.000000
50%          0.000000
75%          1.000000
max        146.000000
Name: business_days_late, dtype: float64


## target


In [3]:
# 영업일 기준으로 5일을 초과해 늦게 결제된 경우
# business_days_late > 5 면 1 아니면 0
train_featured['target'] = (train_featured['business_days_late'] > 5).astype(int)
test_featured['target'] = (test_featured['business_days_late'] > 5).astype(int)

# 기존 target_old와 target의 분포 확인해보기
print(pd.DataFrame({
    'train_target_old': train_featured['target_old'].value_counts().sort_index(),
    'train_target': train_featured['target'].value_counts().sort_index(),
    'test_target_old': test_featured['target_old'].value_counts().sort_index(),
    'test_target': test_featured['target'].value_counts().sort_index(),
}).fillna(0).astype(int))

   train_target_old  train_target  test_target_old  test_target
0             18393         29870             4843         7567
1             13607          2130             3157          433


## due_weekend_flag


In [4]:
# due_in_date가 토요일 또는 일요일이면 1, 평일이면 0
train_featured['due_weekend_flag'] = train_featured['due_in_date'].dt.weekday.isin([5, 6]).astype(int)
test_featured['due_weekend_flag'] = test_featured['due_in_date'].dt.weekday.isin([5, 6]).astype(int)

print('train:', train_featured['due_weekend_flag'].value_counts().sort_index().to_dict())
print('test :', test_featured['due_weekend_flag'].value_counts().sort_index().to_dict())

train: {0: 23110, 1: 8890}
test : {0: 5773, 1: 2227}


## business_day_gap


In [5]:
# 만기일이 다음 영업일까지 얼마나 떨어져 있는가?
def add_business_day_gap(df: pd.DataFrame) -> None:
    due_weekday = df['due_in_date'].dt.weekday

    # 토요일 만기 = 2일, 일요일 만기= 1일
    df['business_day_gap'] = np.select(
        [due_weekday.eq(5), due_weekday.eq(6)],
        [2, 1],
        default=0,
    ).astype(int)

add_business_day_gap(train_featured)
add_business_day_gap(test_featured)

print('train:', train_featured['business_day_gap'].value_counts().sort_index().to_dict())
print('test :', test_featured['business_day_gap'].value_counts().sort_index().to_dict())

train: {0: 23110, 1: 3831, 2: 5059}
test : {0: 5773, 1: 960, 2: 1267}


## amount_bin


In [6]:
# amount_in_usd를 train 데이터에서4분위 구간 분할
_, raw_amount_bins = pd.qcut(
    train_featured['amount_in_usd'],
    q=4,
    labels=False,
    retbins=True,
    duplicates='drop',
)

amount_bins = np.r_[-np.inf, raw_amount_bins[1:-1], np.inf]
amount_labels = list(range(len(amount_bins) - 1))

train_featured['amount_bin'] = pd.cut(
    train_featured['amount_in_usd'],
    bins=amount_bins,
    labels=amount_labels,
    include_lowest=True,
).astype('int64')

test_featured['amount_bin'] = pd.cut(
    test_featured['amount_in_usd'],
    bins=amount_bins,
    labels=amount_labels,
    include_lowest=True,
).astype('int64')

print('Amount bin edges:', amount_bins)
print('train:', train_featured['amount_bin'].value_counts().sort_index().to_dict())
print('test :', test_featured['amount_bin'].value_counts().sort_index().to_dict())

Amount bin edges: [       -inf  8.4071601   9.73095483 10.72198876         inf]
train: {0: 8000, 1: 8000, 2: 8000, 3: 8000}
test : {0: 1901, 1: 1975, 2: 2008, 3: 2116}


## cust_allowed_pay_days_late_rate_past


In [7]:
feature = 'cust_allowed_pay_days_late_rate_past'
train_featured[feature] = 0.0
test_featured[feature] = 0.0

# 결제 완료된 송장만
for cust, current_rows_for_customer in train_featured.groupby('cust_number', sort=False):
    customer_history = train_featured[train_featured['cust_number'].eq(cust)]

    for current_date, current_rows in current_rows_for_customer.groupby('baseline_create_date', sort=True):
        paid = customer_history[
            (customer_history['baseline_create_date'] < current_date)
            & (customer_history['clear_date'] < current_date)
        ]
        if paid.empty:
            continue

        # 같은 고객 + 같은 Allowed_Pay_Days의 과거 연체율
        rate_by_allowed_days = paid.groupby('Allowed_Pay_Days')['target'].mean()
        train_featured.loc[current_rows.index, feature] = (
            current_rows['Allowed_Pay_Days'].map(rate_by_allowed_days).fillna(0.0).astype(float)
        )

# test는 train 전체 + 현재보다 과거인 test 이력만
test_history = pd.concat([train_featured, test_featured], ignore_index=True)
for cust, current_rows_for_customer in test_featured.groupby('cust_number', sort=False):
    customer_history = test_history[test_history['cust_number'].eq(cust)]

    for current_date, current_rows in current_rows_for_customer.groupby('baseline_create_date', sort=True):
        paid = customer_history[
            (customer_history['baseline_create_date'] < current_date)
            & (customer_history['clear_date'] < current_date)
        ]
        if paid.empty:
            continue

        rate_by_allowed_days = paid.groupby('Allowed_Pay_Days')['target'].mean()
        test_featured.loc[current_rows.index, feature] = (
            current_rows['Allowed_Pay_Days'].map(rate_by_allowed_days).fillna(0.0).astype(float)
        )

print(train_featured[feature].describe())

count    32000.000000
mean         0.049603
std          0.160327
min          0.000000
25%          0.000000
50%          0.003726
75%          0.017668
max          1.000000
Name: cust_allowed_pay_days_late_rate_past, dtype: float64


## ratio_paid_invoices_late_past


In [8]:
feature = 'ratio_paid_invoices_late_past'
train_featured[feature] = 0.0
test_featured[feature] = 0.0

# 같은 고객의 과거 결제 완료 송장 중 target 기준 연체 비율
for cust, current_rows_for_customer in train_featured.groupby('cust_number', sort=False):
    customer_history = train_featured[train_featured['cust_number'].eq(cust)]

    for current_date, current_rows in current_rows_for_customer.groupby('baseline_create_date', sort=True):
        paid = customer_history[
            (customer_history['baseline_create_date'] < current_date)
            & (customer_history['clear_date'] < current_date)
        ]
        if not paid.empty:
            train_featured.loc[current_rows.index, feature] = float(paid['target'].mean())

test_history = pd.concat([train_featured, test_featured], ignore_index=True)
for cust, current_rows_for_customer in test_featured.groupby('cust_number', sort=False):
    customer_history = test_history[test_history['cust_number'].eq(cust)]

    for current_date, current_rows in current_rows_for_customer.groupby('baseline_create_date', sort=True):
        paid = customer_history[
            (customer_history['baseline_create_date'] < current_date)
            & (customer_history['clear_date'] < current_date)
        ]
        if not paid.empty:
            test_featured.loc[current_rows.index, feature] = float(paid['target'].mean())

print(train_featured[feature].describe())

count    32000.000000
mean         0.050203
std          0.159811
min          0.000000
25%          0.000000
50%          0.004625
75%          0.019608
max          1.000000
Name: ratio_paid_invoices_late_past, dtype: float64


## avg_days_late_paid_late_past


In [9]:
feature = 'avg_days_late_paid_late_past'
train_featured[feature] = 0.0
test_featured[feature] = 0.0

# 같은 고객의 과거 연체 송장만 대상으로 평균 영업일 지연일
for cust, current_rows_for_customer in train_featured.groupby('cust_number', sort=False):
    customer_history = train_featured[train_featured['cust_number'].eq(cust)]

    for current_date, current_rows in current_rows_for_customer.groupby('baseline_create_date', sort=True):
        paid_late = customer_history[
            (customer_history['baseline_create_date'] < current_date)
            & (customer_history['clear_date'] < current_date)
            & (customer_history['target'].eq(1))
        ]
        if not paid_late.empty:
            train_featured.loc[current_rows.index, feature] = float(paid_late['business_days_late'].mean())

test_history = pd.concat([train_featured, test_featured], ignore_index=True)
for cust, current_rows_for_customer in test_featured.groupby('cust_number', sort=False):
    customer_history = test_history[test_history['cust_number'].eq(cust)]

    for current_date, current_rows in current_rows_for_customer.groupby('baseline_create_date', sort=True):
        paid_late = customer_history[
            (customer_history['baseline_create_date'] < current_date)
            & (customer_history['clear_date'] < current_date)
            & (customer_history['target'].eq(1))
        ]
        if not paid_late.empty:
            test_featured.loc[current_rows.index, feature] = float(paid_late['business_days_late'].mean())

print(train_featured[feature].describe())

count    32000.000000
mean        13.934218
std         17.427254
min          0.000000
25%          0.000000
50%          8.000000
75%         18.000000
max         86.000000
Name: avg_days_late_paid_late_past, dtype: float64


## sum_outstanding_amount_past


In [10]:
feature = 'sum_outstanding_amount_past'
train_featured[feature] = 0.0
test_featured[feature] = 0.0

# 아직 결제되지 않은 과거 송장의 총금액
for cust, current_rows_for_customer in train_featured.groupby('cust_number', sort=False):
    customer_history = train_featured[train_featured['cust_number'].eq(cust)]

    for current_date, current_rows in current_rows_for_customer.groupby('baseline_create_date', sort=True):
        outstanding = customer_history[
            (customer_history['baseline_create_date'] < current_date)
            & (customer_history['clear_date'] >= current_date)
        ]
        train_featured.loc[current_rows.index, feature] = max(float(outstanding['amount_in_usd'].sum()), 0.0)

test_history = pd.concat([train_featured, test_featured], ignore_index=True)
for cust, current_rows_for_customer in test_featured.groupby('cust_number', sort=False):
    customer_history = test_history[test_history['cust_number'].eq(cust)]

    for current_date, current_rows in current_rows_for_customer.groupby('baseline_create_date', sort=True):
        outstanding = customer_history[
            (customer_history['baseline_create_date'] < current_date)
            & (customer_history['clear_date'] >= current_date)
        ]
        test_featured.loc[current_rows.index, feature] = max(float(outstanding['amount_in_usd'].sum()), 0.0)

print(train_featured[feature].describe())

count    32000.000000
mean       740.875133
std       1071.324304
min          0.000000
25%         35.262890
50%        162.661170
75%        650.793740
max       3635.495781
Name: sum_outstanding_amount_past, dtype: float64


## recent_5_late_rate


In [11]:
feature = 'recent_5_late_rate'
train_featured[feature] = 0.0
test_featured[feature] = 0.0

# 최근 5건 결제 완료 송장 중 연체 비율
for cust, current_rows_for_customer in train_featured.groupby('cust_number', sort=False):
    customer_history = train_featured[train_featured['cust_number'].eq(cust)].copy()
    customer_history['_eligible_paid_date'] = customer_history[['baseline_create_date', 'clear_date']].max(axis=1)

    for current_date, current_rows in current_rows_for_customer.groupby('baseline_create_date', sort=True):
        paid = customer_history[
            (customer_history['baseline_create_date'] < current_date)
            & (customer_history['clear_date'] < current_date)
        ]
        if paid.empty:
            continue

        recent_paid = paid.sort_values(['_eligible_paid_date', 'baseline_create_date', 'clear_date']).tail(5)
        train_featured.loc[current_rows.index, feature] = float(recent_paid['target'].mean())

test_history = pd.concat([train_featured, test_featured], ignore_index=True)
for cust, current_rows_for_customer in test_featured.groupby('cust_number', sort=False):
    customer_history = test_history[test_history['cust_number'].eq(cust)].copy()
    customer_history['_eligible_paid_date'] = customer_history[['baseline_create_date', 'clear_date']].max(axis=1)

    for current_date, current_rows in current_rows_for_customer.groupby('baseline_create_date', sort=True):
        paid = customer_history[
            (customer_history['baseline_create_date'] < current_date)
            & (customer_history['clear_date'] < current_date)
        ]
        if paid.empty:
            continue

        recent_paid = paid.sort_values(['_eligible_paid_date', 'baseline_create_date', 'clear_date']).tail(5)
        test_featured.loc[current_rows.index, feature] = float(recent_paid['target'].mean())

print(train_featured[feature].describe())

count    32000.000000
mean         0.041245
std          0.160203
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000
Name: recent_5_late_rate, dtype: float64


## csv 저장


In [12]:
train_featured.to_csv(TRAIN_OUT, index=False)
test_featured.to_csv(TEST_OUT, index=False)

print('Saved:', TRAIN_OUT)
print('Saved:', TEST_OUT)
print('Train output exists:', TRAIN_OUT.exists(), 'size:', TRAIN_OUT.stat().st_size)
print('Test output exists :', TEST_OUT.exists(), 'size:', TEST_OUT.stat().st_size)

Saved: /content/output/train_eda.csv
Saved: /content/output/test_eda.csv
Train output exists: True size: 5696379
Test output exists : True size: 1503809
